In [1]:
%pip install -Uq "unstructured[all-docs]"
%pip install -Uq langchain_chroma
%pip install -Uq langchain langchain-community langchain-openai
%pip install -Uq python_dotenv


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json 
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain Components
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.messages import HumanMessage

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
def partition_document(file_path: str):
    """Extract elements from PDF using Unstructured"""

    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res", # Most accurate (but slower) processing method - uses (layout detection models, OCRs & table transformers etc.)
        infer_table_structure=True, # Keep tables as structured HTML
        extract_image_block_types=["Image"], # Grab images found in PDF,
        extract_image_block_to_payload=True # Store images as base64 data
    )

    print(f"Extracted {len(elements)} elements")
    return elements

file_path = "./docs/amazon-dynamo.pdf"
elements = partition_document(file_path)


The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


Extracted 255 elements


In [7]:
# All types of different atomic elements we see from unstructured
set([str(type(ele)) for ele in elements])

{"<class 'unstructured.documents.elements.FigureCaption'>",
 "<class 'unstructured.documents.elements.Footer'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [21]:
#elements[100].to_dict() # Sample Image Element
tables = [ele for ele in elements if ele.category == "Table"]
tables[0].to_dict() # Sample Table Element

{'type': 'Table',
 'element_id': '9255e65385eca9f0f58343e60504721b',
 'text': 'Problem Technique Advantage Partitioning Consistent Hashing Incremental Scalability High Availability Vector clocks with Version size is for writes reconciliation during decoupled from reads update rates. Handling temporary Sloppy Quorum and Provides high failures hinted handoff availability and durability guarantee when some of the replicas are not available. Recovering from Anti-entropy using Synchronizes permanent failures Merkle trees divergent replicas in the background. Membership and Gossip-based Preserves symmetry failure detection membership protocol and avoids having a and failure detection. centralized registry for storing membership and node liveness information.',
 'metadata': {'detection_class_prob': 0.9313226342201233,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(890.712646484375),
     np.float64(268.3202819824219)),
    (np.float64(890.712646484375), np.float64(942.4926

In [22]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("Creating smart chunks...")

    chunks = chunk_by_title(
        elements,
        max_characters=3000, # Hard Limit - Never exceed 3000 chars per chunk
        new_after_n_chars=2400, # Try to start new chunk after 2400
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbours
    )

    print(f"Created {len(chunks)} chunks")
    return chunks

chunks = create_chunks_by_title(elements)

Creating smart chunks...
Created 49 chunks


In [25]:
# Single Chunk
#chunks[0].to_dict()

# View original elements
chunks[0].metadata.orig_elements

In [27]:
def separate_content_types(chunk):
    """Analyise what types of content are in chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }

    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__

            # Handle Tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            # Handle Images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
        
        
    content_data['types'] = list(set(content_data['types']))
    return content_data

def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-Enhanced Summary for mixed content"""

    try:
        # Initialise LLM 
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

        prompt_text = f"""You are creating a searchable description for document content retrieval.
        CONTENT TO ANALISE:
        TEXT CONTENT:
        {text}
        """

        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"

                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers, and data points from the text and tables
                2. Main topics and concepts discussed
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:"""
        
        message_content = [{"type": "text", "text": prompt_text}]

        # Add images to the message
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })

        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        return response.content
    except Exception as e:
        print(f"  AI Summary Failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary




def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print(" Processing chunks with AI Summaries")

    langchain_documents = []
    total_chunks = len(chunks)

    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f" Processing chunk {current_chunk}/{total_chunks}")

        # Analyse chunk data
        content_data = separate_content_types(chunk)
        print(f" Types found: {content_data}")
        print(f" Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")

        if content_data['tables'] or content_data['images']:
            print(f" Creating AI Summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'],
                    content_data['images']
                )
                print(f" AI Summary created successfully")
                print(f" Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f" AI Summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f" Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        langchain_documents.append(doc)
    print(f" Proccessed {len(langchain_documents)} chunks")
    return langchain_documents


processed_chunks = summarise_chunks(chunks)

 Processing chunks with AI Summaries
 Processing chunk 1/49
 Types found: {'text': 'Dynamo: Amazon’s Highly Available Key-value Store\n\nGiuseppe DeCandia, Deniz Hastorun, Madan Jampani, Gunavardhan Kakulapati, Avinash Lakshman, Alex Pilchin, Swaminathan Sivasubramanian, Peter Vosshall and Werner Vogels\n\nAmazon.com\n\nABSTRACT\n\nReliability at massive scale is one of the biggest challenges we face at Amazon.com, one of the largest e-commerce operations in the world; even the slightest outage has significant financial consequences and impacts customer trust. The Amazon.com platform, which provides services for many web sites worldwide, is implemented on top of an infrastructure of tens of thousands of servers and network components located in many datacenters around the world. At this scale, small and large components fail continuously and the way persistent state is managed in the face of these failures drives the reliability and scalability of the software systems.\n\nOne of the le

In [28]:
processed_chunks

[Document(metadata={'original_content': '{"raw_text": "Dynamo: Amazon\\u2019s Highly Available Key-value Store\\n\\nGiuseppe DeCandia, Deniz Hastorun, Madan Jampani, Gunavardhan Kakulapati, Avinash Lakshman, Alex Pilchin, Swaminathan Sivasubramanian, Peter Vosshall and Werner Vogels\\n\\nAmazon.com\\n\\nABSTRACT\\n\\nReliability at massive scale is one of the biggest challenges we face at Amazon.com, one of the largest e-commerce operations in the world; even the slightest outage has significant financial consequences and impacts customer trust. The Amazon.com platform, which provides services for many web sites worldwide, is implemented on top of an infrastructure of tens of thousands of servers and network components located in many datacenters around the world. At this scale, small and large components fail continuously and the way persistent state is managed in the face of these failures drives the reliability and scalability of the software systems.\\n\\nOne of the lessons our org

In [31]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

✅ Exported 49 chunks to chunks_export.json


In [32]:
def create_vector_store(documents, persist_directory="db/chroma_db"):
    """Create and persist ChromaDB Vector Store"""
    print(" Creating embeddings and storing in ChromaDB")
    
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

    #Create ChromaDB Vector Store
    print("--- Creating Vector Store ---")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space" : "cosine"}
    )
    print("--- Finished Creating Vector Store ---")

    print(f"Vector store created and save to {persist_directory}")
    return vectorstore

db = create_vector_store(processed_chunks)


 Creating embeddings and storing in ChromaDB
--- Creating Vector Store ---
--- Finished Creating Vector Store ---
Vector store created and save to db/chroma_db


In [33]:
# Retrieval

query = """For forming a performance oriented SLA is to describe it using average, median and expected 
variance - are these metrics good enough ?"""

retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

export_chunks_to_json(chunks, "rag_result.json")

✅ Exported 3 chunks to rag_result.json


[{'chunk_id': 1,
  'enhanced_content': '207\n\nproduction systems have shown that this approach provides a better overall experience compared to those systems that meet SLAs defined based on the mean or median.\n\nIn this paper there are many references to this 99.9th percentile of distributions, which reflects Amazon engineers’ relentless focus on performance from the perspective of the customers’ experience. Many papers report on averages, so these are included where it makes sense for comparison purposes. Nevertheless, Amazon’s engineering and optimization efforts are not focused on averages. Several techniques, such as the load balanced selection of write coordinators, are purely targeted at controlling performance at the 99.9th percentile.\n\nStorage systems often play an important role in establishing a service’s SLA, especially if the business logic is relatively lightweight, as is the case for many Amazon services. State management then becomes the main component of a service’s

In [39]:
def run_complete_ingestion_pipleline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("Starting RAG Ingestion Pipeline")
    print("="*50)

    # Step 1: Partition
    elements = partition_document(pdf_path)

    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)

    # Step 3: AI Summarisation
    summarised_chunks = summarise_chunks(chunks)

    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks, persist_directory="dbv2/chroma_db")

    print("Pipline Completed Successfully!")
    return db




In [40]:
db = run_complete_ingestion_pipleline("./docs/amazon-dynamo.pdf")

Starting RAG Ingestion Pipeline
Extracted 255 elements
Creating smart chunks...
Created 49 chunks
 Processing chunks with AI Summaries
 Processing chunk 1/49
 Types found: {'text': 'Dynamo: Amazon’s Highly Available Key-value Store\n\nGiuseppe DeCandia, Deniz Hastorun, Madan Jampani, Gunavardhan Kakulapati, Avinash Lakshman, Alex Pilchin, Swaminathan Sivasubramanian, Peter Vosshall and Werner Vogels\n\nAmazon.com\n\nABSTRACT\n\nReliability at massive scale is one of the biggest challenges we face at Amazon.com, one of the largest e-commerce operations in the world; even the slightest outage has significant financial consequences and impacts customer trust. The Amazon.com platform, which provides services for many web sites worldwide, is implemented on top of an infrastructure of tens of thousands of servers and network components located in many datacenters around the world. At this scale, small and large components fail continuously and the way persistent state is managed in the face 

In [46]:
def generate_final_answer(chunks, query):
    """Generate Final Answer using Multi-Modal Content"""

    try: 
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        prompt_text = f"""Based on the following documents, please answer this questions: {query}
        CONTENT TO ANALYSE:
        """

        for i, chunk in enumerate(chunks):
            prompt_text += f"--- Document {i+1} ---\n"

            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])

                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"

                tables_html = original_data.get("tables_html", [])
                if tables_html:
                    prompt_text += "TABLES:\n"
                    for j, table in enumerate(tables_html):
                        prompt_text += f"Table {j+1}:\n{table}\n\n"
                
            prompt_text += "\n"

        prompt_text += """
        Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents."
        ANSWER:"""

        message_content = [{"type": "text", "text": prompt_text}]

        for chunk in chunks:
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                images_base64 = original_data.get("images_base64", [])

                for image_base64 in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image_base64}"
                        }
                    })
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        return response.content
    except Exception as e:
        print(f" Answer generation failed: {e}")
        return "Sorry, I encountered an error while generating the answer"

In [49]:
# Usage: 
query = """For forming a performance oriented SLA is to describe it using average, median and expected 
variance - are these metrics good enough ?"""

# query = "What operations do Dynamo exposes ?"

# query = "What is Operating System?"

retriever = db.as_retriever(search_kwargs={"k": 3})
chunks = retriever.invoke(query)

final_answer = generate_final_answer(chunks, query)
print(final_answer)

I don't have enough information to answer that question based on the provided documents.
